In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np

# ── Data ──────────────────────────────────────────────────────────────────────
df = pd.read_excel(
    '/mnt/user-data/uploads/1781788381570_June_to_Nov_actual_sales.xlsx',
    sheet_name='MoM_Sales'
)

# ── Helper ────────────────────────────────────────────────────────────────────
def to_lakhs(val):
    """Convert raw number to '42.11 L' string."""
    return f"{val / 100_000:.2f} L"

# ── Plot order & palette ───────────────────────────────────────────────────────
month_order = ['FY_Total', 'June', 'July', 'August', 'September', 'October', 'November']

colors = {
    'FY_Total'  : '#1f4e79',   # deep navy  — stands out as the summary line
    'June'      : '#2e86ab',
    'July'      : '#e84855',
    'August'    : '#f18f01',
    'September' : '#5c4b8a',
    'October'   : '#3bb273',
    'November'  : '#c94040',
}

markers = {
    'FY_Total'  : 'D',
    'June'      : 'o',
    'July'      : 's',
    'August'    : '^',
    'September' : 'P',
    'October'   : 'X',
    'November'  : 'v',
}

# ── Figure ────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 7))
fig.patch.set_facecolor('#f9f9f9')
ax.set_facecolor('#f9f9f9')

fy_labels = ['FY\'23', 'FY\'24', 'FY\'25', 'FY\'26']
x = np.arange(len(fy_labels))

for month in month_order:
    subset = df[df['MONTH_NAME'] == month].copy()
    subset = subset.set_index('FINANCIAL_YEAR').reindex(fy_labels)
    y = subset['ACTUAL_SALES'].values

    lw  = 2.8 if month == 'FY_Total' else 1.8
    ls  = '--' if month == 'FY_Total' else '-'
    ms  = 9   if month == 'FY_Total' else 7
    zord = 5  if month == 'FY_Total' else 3

    ax.plot(x, y,
            label=month,
            color=colors[month],
            linewidth=lw,
            linestyle=ls,
            marker=markers[month],
            markersize=ms,
            zorder=zord)

    # ── Annotations ───────────────────────────────────────────────────────────
    for xi, yi in zip(x, y):
        if pd.isna(yi):
            continue
        # nudge label above the point; FY_Total gets a bit more space
        offset = 18 if month == 'FY_Total' else 10
        ax.annotate(
            to_lakhs(yi),
            xy=(xi, yi),
            xytext=(0, offset),
            textcoords='offset points',
            ha='center',
            va='bottom',
            fontsize=7.5,
            color=colors[month],
            fontweight='bold' if month == 'FY_Total' else 'normal'
        )

# ── Axes formatting ────────────────────────────────────────────────────────────
ax.set_xticks(x)
ax.set_xticklabels(fy_labels, fontsize=12, fontweight='bold')
ax.set_xlabel('Financial Year', fontsize=13, labelpad=10)
ax.set_ylabel('Actual Sales (in Lakhs)', fontsize=13, labelpad=10)
ax.set_title(
    'Month-wise Actual Sales Trend : FY\'23 to FY\'26',
    fontsize=15, fontweight='bold', pad=18
)

# Y-axis in lakhs
ax.yaxis.set_major_formatter(
    ticker.FuncFormatter(lambda val, _: f"{val/100_000:.0f} L")
)

ax.set_xlim(-0.3, 3.3)
ax.margins(y=0.18)   # breathing room for top annotations
ax.grid(axis='y', linestyle='--', alpha=0.4, color='grey')
ax.spines[['top', 'right']].set_visible(False)

# ── Legend ────────────────────────────────────────────────────────────────────
legend = ax.legend(
    title='Month / Period',
    title_fontsize=10,
    fontsize=9,
    loc='upper left',
    framealpha=0.85,
    edgecolor='#cccccc',
    ncol=2
)

plt.tight_layout()
plt.savefig('/mnt/user-data/outputs/mom_sales_chart.png', dpi=180, bbox_inches='tight')
print("Saved.")